In [1]:
import sys, platform
ON_LINUXONE = platform.machine() == "s390x"
DATA_PATH = ("/home/ibm-user/transactions_history.csv" if ON_LINUXONE
             else "transactions_history.csv")
SEED = 42
TRAIN_END_YEAR = 2015         # train: up to 2015
VALID_YEARS = (2016, 2017)    # validation: compare models here
TEST_YEARS = (2018, 2019)     # test: use once in step 22

import pandas as pd
import numpy as np
import sklearn
import xgboost
import tensorflow as tf
import keras

print("Machine:", platform.machine(), "| Python:", sys.version.split()[0])

Machine: s390x | Python: 3.12.15


In [3]:
# 1. Load data
df = pd.read_csv(DATA_PATH)

# 2. Clean Amount: "$134.09" -> 134.09
df['Amount_num'] = df['Amount'].astype(str).str.replace('$', '', regex=False).astype(float)

# 3. Clean 'Is Fraud?' -> 1 / 0
# Bypass the object dtype check to support Pandas 3.x StringDtype
df['is_fraud'] = (df['Is Fraud?'].astype(str).str.strip().str.lower() == 'yes').astype(int)

# 4. Create single timestamp 'ts'
time_split = df['Time'].astype(str).str.split(':', expand=True)
df['hour'] = time_split[0].astype(int)
df['minute'] = time_split[1].astype(int)

df['ts'] = pd.to_datetime(dict(
    year=df['Year'],
    month=df['Month'],
    day=df['Day'],
    hour=df['hour'],
    minute=df['minute']
))

# 5. Sort chronologically
df = df.sort_values(by=['User', 'Card', 'ts']).reset_index(drop=True)

# 6. Assign train / valid / test splits by year
def assign_split(year):
    if year <= TRAIN_END_YEAR:
        return 'train'
    elif year in VALID_YEARS:
        return 'valid'
    elif year in TEST_YEARS:
        return 'test'
    else:
        return 'drop'

df['split'] = df['Year'].apply(assign_split)

# 7. Drop 2020+
df = df[df['split'] != 'drop'].reset_index(drop=True)

# 8. Print fraud counts to verify
fraud_counts = df.groupby('split')['is_fraud'].sum()
print("--- Fraud Count per Split ---")
print(fraud_counts)

--- Fraud Count per Split ---
split
test      60
train    178
valid     33
Name: is_fraud, dtype: int64


In [4]:
from sklearn.metrics import precision_score, recall_score, precision_recall_curve, auc, confusion_matrix

def report(y_true, scores, threshold=0.5):
    # Convert probabilities to binary predictions based on the threshold
    preds = (scores >= threshold).astype(int)
    
    # Calculate Precision and Recall
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)
    
    # Calculate PR-AUC (Precision-Recall Area Under Curve)
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, scores)
    pr_auc = auc(recall_curve, precision_curve)
    
    # Calculate False Positive Rate (FPR) and Alerts per 1,000
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    alerts_per_1000 = ((tp + fp) / len(y_true)) * 1000
    
    print(f"Precision:           {precision:.4f}")
    print(f"Recall:              {recall:.4f}")
    print(f"PR-AUC:              {pr_auc:.4f}")
    print(f"False Positive Rate: {fpr:.6f}")
    print(f"Alerts per 1,000:    {alerts_per_1000:.2f}")
    
    return precision, recall, pr_auc, fpr, alerts_per_1000

# Test the function with random data to verify it works
np.random.seed(SEED)
dummy_y = np.random.randint(0, 2, 10000)
dummy_scores = np.random.rand(10000)
print("--- Test Report (Random Scores) ---")
_ = report(dummy_y, dummy_scores, threshold=0.5)

--- Test Report (Random Scores) ---
Precision:           0.5040
Recall:              0.4965
PR-AUC:              0.4986
False Positive Rate: 0.486136
Alerts per 1,000:    491.30


In [5]:
# 1. Base numerical and time features
df['amount'] = df['Amount_num']
df['weekday'] = df['ts'].dt.dayofweek
df['is_refund'] = (df['amount'] < 0).astype(int)

# 2. Chip / Swipe / Online (One-hot encode)
chip_dummies = pd.get_dummies(df['Use Chip'], prefix='tx_type', dtype=int)
df = pd.concat([df, chip_dummies], axis=1)

# 3. Foreign Merchant (US states are 2 letters; everything else is foreign)
df['foreign_merchant'] = (~df['Merchant State'].astype(str).str.match(r'^[A-Za-z]{2}$', na=False)).astype(int)

# 4. Errors (One-hot encode comma-separated error types)
df['Errors?'] = df['Errors?'].fillna('')
errors_dummies = df['Errors?'].str.get_dummies(sep=',').add_prefix('err_')
df = pd.concat([df, errors_dummies], axis=1)

# 5. Define the final feature list
FEATURE_COLUMNS = ['amount', 'hour', 'weekday', 'is_refund', 'foreign_merchant', 'MCC'] + \
                  list(chip_dummies.columns) + list(errors_dummies.columns)

print("--- Feature Columns ---")
for col in FEATURE_COLUMNS:
    print(col)

print("\nFeature table shape:", df[FEATURE_COLUMNS].shape)

--- Feature Columns ---
amount
hour
weekday
is_refund
foreign_merchant
MCC
tx_type_Chip Transaction
tx_type_Online Transaction
tx_type_Swipe Transaction
err_Bad CVV
err_Bad Card Number
err_Bad Expiration
err_Bad PIN
err_Bad Zipcode
err_Insufficient Balance
err_Technical Glitch

Feature table shape: (242493, 16)


In [6]:
# Ensure data is strictly sorted chronologically per card before looking backwards
df = df.sort_values(by=['User', 'Card', 'ts']).reset_index(drop=True)

# We need a time index to use Pandas 'rolling' windows
df_time = df.set_index('ts')
grouped_time = df_time.groupby(['User', 'Card'])

# 1. Velocity Features: Rolling counts and sums over time
df['count_1h'] = grouped_time['amount'].transform(lambda x: x.rolling('1h').count()).reset_index(drop=True)
df['count_24h'] = grouped_time['amount'].transform(lambda x: x.rolling('24h').count()).reset_index(drop=True)
df['count_7d'] = grouped_time['amount'].transform(lambda x: x.rolling('7d').count()).reset_index(drop=True)
df['amt_24h'] = grouped_time['amount'].transform(lambda x: x.rolling('24h').sum()).reset_index(drop=True)

# 2. Minutes since the previous transaction
df['mins_since_last'] = df.groupby(['User', 'Card'])['ts'].diff().dt.total_seconds() / 60.0
df['mins_since_last'] = df['mins_since_last'].fillna(0) # First transaction has 0 gap

# 3. Behavior Features: Median ratio and new locations
df['median_so_far'] = df.groupby(['User', 'Card'])['amount'].transform(lambda x: x.expanding().median())
df['ratio_median'] = np.where(df['median_so_far'] > 0, df['amount'] / df['median_so_far'], 0)

# `duplicated` returns True for everything AFTER the first appearance. 
# By using `~` (NOT), we flag the VERY FIRST appearance with a 1.
df['first_time_merchant'] = (~df.duplicated(subset=['User', 'Card', 'Merchant Name'])).astype(int)
df['first_time_city'] = (~df.duplicated(subset=['User', 'Card', 'Merchant City'])).astype(int)

# Add to feature list
NEW_FEATURES = ['count_1h', 'count_24h', 'count_7d', 'amt_24h', 'mins_since_last', 
                'ratio_median', 'first_time_merchant', 'first_time_city']
FEATURE_COLUMNS.extend(NEW_FEATURES)

# --- LEAKAGE CHECK ---
# We test if the calculation changes if we only had half the data.
print("--- Running Leakage Check ---")
user_0, card_0 = df['User'].iloc[0], df['Card'].iloc[0]
card_df = df[(df['User'] == user_0) & (df['Card'] == card_0)].copy()
half_idx = len(card_df) // 2

# Calculate median on JUST the first half of the data
first_half = card_df.iloc[:half_idx].copy()
first_half_median = first_half['amount'].expanding().median()

# Compare it to the first half of the full calculation
diff = (first_half_median - card_df['median_so_far'].iloc[:half_idx]).abs().max()

if diff == 0:
    print("Leakage check PASSED! (0 difference, no future data leaked into the past)")
else:
    print(f"Leakage check FAILED! Max difference: {diff}")
    
print("\nTotal features:", len(FEATURE_COLUMNS))

/tmp/ipykernel_478/2952728577.py:11: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  df['count_7d'] = grouped_time['amount'].transform(lambda x: x.rolling('7d').count()).reset_index(drop=True)
/tmp/ipykernel_478/2952728577.py:11: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  df['count_7d'] = grouped_time['amount'].transform(lambda x: x.rolling('7d').count()).reset_index(drop=True)
/tmp/ipykernel_478/2952728577.py:11: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  df['count_7d'] = grouped_time['amount'].transform(lambda x: x.rolling('7d').count()).reset_index(drop=True)
/tmp/ipykernel_478/2952728577.py:11: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  df['count_7d'] = grouped_time['amount'].transform(lambda x: x.rolling('7d').count()).reset_index(drop=True)
/tmp/ipykernel_4

--- Running Leakage Check ---
Leakage check PASSED! (0 difference, no future data leaked into the past)

Total features: 24
